In [ ]:
"""
One-cell data preparation notebook.

Why helper functions in a one-cell notebook?
- They keep each step isolated and easier to debug during a live demo.
- They make the execution flow readable: load -> chunk -> embed -> build collection.
- They avoid repeating logic if you tweak chunk size/model later.
"""

from __future__ import annotations

import gc
import shutil
import time
from pathlib import Path
from typing import List

import numpy as np
import PyPDF2
from sentence_transformers import SentenceTransformer
import zvec

# Demo configuration
PDF_PATH = Path("pdf_doc/NFL_2025.pdf")
COLLECTION_PATH = Path("nfl_2025_collection")
COLLECTION_NAME = "nfl_2025_collection"
MODEL_NAME = "all-MiniLM-L6-v2"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
REBUILD_COLLECTION = True


def read_pdf_text(pdf_path: Path) -> str:
    """Read all pages and return one combined text string."""
    if not pdf_path.exists():
        raise FileNotFoundError(f"Missing required file: {pdf_path}")

    pages: List[str] = []
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            pages.append(page.extract_text() or "")

    text = "\n".join(pages).strip()
    if not text:
        raise ValueError("PDF was read successfully, but extracted text is empty.")
    return text


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    """Split text into overlapping chunks for better semantic retrieval."""
    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")

    step = chunk_size - overlap
    chunks: List[str] = []

    for start in range(0, len(text), step):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

    if not chunks:
        raise ValueError("Chunking produced no chunks.")
    return chunks


def embed_chunks(chunks: List[str], model_name: str) -> np.ndarray:
    """Generate normalized embeddings once so later query tests are fast."""
    model = SentenceTransformer(model_name)
    vectors = model.encode(chunks, show_progress_bar=False, normalize_embeddings=True)
    return np.asarray(vectors, dtype=np.float32)


def build_collection(path: Path, name: str, vectors: np.ndarray, chunks: List[str], rebuild: bool = True):
    """Create/open a ZVec collection and insert all docs + vectors."""
    zvec.init(log_type=zvec.LogType.CONSOLE, log_level=zvec.LogLevel.WARN)

    if rebuild and path.exists():
        shutil.rmtree(path)

    text_field = zvec.FieldSchema(
        name="text",
        data_type=zvec.DataType.STRING,
        nullable=False,
    )
    embedding_vector = zvec.VectorSchema(
        name="embedding",
        data_type=zvec.DataType.VECTOR_FP32,
        dimension=int(vectors.shape[1]),
        index_param=zvec.HnswIndexParam(metric_type=zvec.MetricType.COSINE),
    )

    schema = zvec.CollectionSchema(
        name=name,
        fields=[text_field],
        vectors=[embedding_vector],
    )

    collection = zvec.create_and_open(path=str(path), schema=schema)

    for idx, (chunk, vector) in enumerate(zip(chunks, vectors)):
        doc = zvec.Doc(
            id=str(idx),
            fields={"text": chunk},
            vectors={"embedding": vector.tolist()},
        )
        collection.insert(doc)

    return collection


# One-click execution
start = time.perf_counter()

raw_text = read_pdf_text(PDF_PATH)
chunks = chunk_text(raw_text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
vectors = embed_chunks(chunks, model_name=MODEL_NAME)
collection = build_collection(
    path=COLLECTION_PATH,
    name=COLLECTION_NAME,
    vectors=vectors,
    chunks=chunks,
    rebuild=REBUILD_COLLECTION,
)

# Flush and release the collection handle so notebook 2 can open without lock conflicts.
collection.flush()
del collection
gc.collect()

elapsed = time.perf_counter() - start
print("READY")
print(f"- PDF: {PDF_PATH}")
print(f"- Chunks: {len(chunks)}")
print(f"- Embedding dim: {vectors.shape[1]}")
print(f"- Collection path: {COLLECTION_PATH}")
print(f"- Total time: {elapsed:.2f} seconds")
print("- Collection handle released for notebook 2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

READY
- PDF: pdf_doc/NFL_2025.pdf
- Chunks: 718
- Embedding dim: 384
- Collection path: nfl_2025_collection
- Total time: 37.82 seconds
- Collection handle released for notebook 2
